<a href="https://colab.research.google.com/github/vad-source/NLPAPP/blob/main/NLPAPP_GEC_Using_Classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## NLP APPLICATIONS
**Designed by:** RAJA VADHANA PRABHAKAR  
**Organization:** BITS PILANI WILP  
**Purpose:** Academic Training / Proof of Concept  

---
#### Attribution & AI Disclosure
- **Original Design:** The logic, Use case design, architecture, and modular structure of this notebook were designed by the author.
- **Development Assistance:** Generative AI (e.g., ChatGPT/Claude/Copilot) was used for coding implementation and debugging support.
- **License:** This work is licensed under the [Apache License 2.0](https://apache.org).

### Grammatical Error Correction (GEC)
1. Load the errorneous input
2. Tokenize the Sentence
3. Encode the tokens into contextual representation
4. Classify the representation to Tags
5. Apply the corrections identified by tags
6. Detokenize and output corrected sentence

 #### **Seq2Seq GEC**

Use case : In chatbots, there are no restrictions on the input received from users, which often leads to the occurrence of many open-class words. Therefore, a broader model is required.

For generation tasks such as rephrasing, fluent correction is necessary, or ideally, a many-to-many model should be used to reorder sentences into coherent text.

Below code uses hugging face model for GEC.

In [ ]:
!pip install -q transformers sentencepiece torch --upgrade

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

In [ ]:
model_name = "prithivida/grammar_error_correcter_v1" # Replace this with any appropriate Hugging Face model of your choice

print("Loading model:", model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

In [ ]:
def seq2seq_correct(sentences, max_length=128, num_beams=4):
    # Accepts a single string or list of strings
    single = isinstance(sentences, str)
    if single:
        sentences = [sentences]
    inputs = tokenizer(sentences, return_tensors="pt", padding=True, truncation=True).to(device)
    outputs = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_length=max_length,
        num_beams=num_beams,
        early_stopping=True
    )
    corrected = [tokenizer.decode(o, skip_special_tokens=True) for o in outputs]
    return corrected[0] if single else corrected

In [ ]:
examples = [
    "She go to school every day.",
    "He have two apple.",
    "I am interesting in machine learning."
]

In [ ]:
for s in examples:
    print("INPUT  :", s)
    print("OUTPUT :", seq2seq_correct(s))
    print("-" * 60)

 #### EditTagger GEC

Use case : In rule engine based support bots, the limited functionality of agents constraints the input received from users, which often needs just closed-class words edits. Therefore, a minimal correction model is may be sufficient.

The below code uses pretrained LM (transformer based) to encode the input and as a post processing a tagger is designed that predicts edit tag per token.

Note: Because the classifier is untrained in this demo, predictions will be essentially random; in practice you would fine tune the classifier (and optionally the encoder) on GEC tag data.


In [ ]:
!pip install -q transformers torch sentencepiece

In [ ]:
#import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel
import numpy as np
import re
import torch.nn.functional as F

In [ ]:
# Config
ENCODER_NAME = "roberta-base"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


In [ ]:

REPLACE_VOCAB = ["goes", "has", "apples"]
TAG_LIST = ["KEEP", "DELETE"] + [f"REPLACE:{w}" for w in REPLACE_VOCAB] + [f"INSERT_BEFORE:{w}" for w in REPLACE_VOCAB]
TAG2IDX = {t:i for i,t in enumerate(TAG_LIST)}
IDX2TAG = {i:t for t,i in TAG2IDX.items()}



In [ ]:
# Model
class EditTagger(nn.Module):
    def __init__(self, encoder_name, num_tags):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(encoder_name)
        hidden = self.encoder.config.hidden_size
        self.classifier = nn.Linear(hidden, num_tags)
    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        last = out.last_hidden_state
        logits = self.classifier(last)
        return logits

tokenizer = AutoTokenizer.from_pretrained(ENCODER_NAME)

# -------------------------
# Canonical word tokenizer (regex) and alignment wrapper
# -------------------------
WORD_RE = re.compile(r"\w+|[^\w\s]", re.UNICODE)


In [ ]:
# HELPER FUNCTIONS
def canonical_word_split(text):
    """
    Deterministic word-level split used for tag alignment.
    Splits words and punctuation as separate tokens.
    """
    return WORD_RE.findall(text.strip())

def tokenize_with_alignment_canonical(text):
    """
    Use canonical_word_split for word-level tokens (the authoritative split for tags),
    then call HF tokenizer with is_split_into_words=True to get subword alignment.
    Returns:
      words (canonical),
      enc (tokenizer output),
      word_ids (list mapping subword idx -> word idx or None)
    """
    words = canonical_word_split(text)
    enc = tokenizer(words, is_split_into_words=True, return_tensors="pt", truncation=True)
    word_ids = enc.word_ids(batch_index=0)
    return words, enc, word_ids

# -------------------------
# Robust aggregation and embedding helpers (same idea as before)
# -------------------------
def aggregate_subword_logits_to_words(logits_subword, word_ids, num_words):
    word_to_logits = {w: [] for w in range(num_words)}
    for i, wid in enumerate(word_ids):
        if wid is None:
            continue
        if 0 <= wid < num_words:
            word_to_logits[wid].append(logits_subword[i].cpu())
    num_tags = logits_subword.size(-1)
    word_logits = []
    for w in range(num_words):
        if len(word_to_logits[w]) == 0:
            word_logits.append(torch.zeros(num_tags))
        else:
            stacked = torch.stack(word_to_logits[w], dim=0)
            word_logits.append(stacked.mean(dim=0))
    return word_logits

def get_word_embeddings_canonical(text, model):
    words, enc, word_ids = tokenize_with_alignment_canonical(text)
    input_ids = enc["input_ids"].to(DEVICE)
    attention_mask = enc["attention_mask"].to(DEVICE)
    with torch.no_grad():
        last = model.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
    hidden = last.size(-1)
    num_words = len(words)
    word_to_vecs = {w: [] for w in range(num_words)}
    for i, wid in enumerate(word_ids):
        if wid is None:
            continue
        if 0 <= wid < num_words:
            word_to_vecs[wid].append(last[0, i].cpu())
    word_vecs = []
    for w in range(num_words):
        if len(word_to_vecs[w]) == 0:
            word_vecs.append(torch.zeros(hidden))
        else:
            stacked = torch.stack(word_to_vecs[w], dim=0)
            word_vecs.append(stacked.mean(dim=0))
    return torch.stack(word_vecs)  # (num_words, H)

# -------------------------
# Reconciliation helper for tags vs canonical words
# -------------------------
def reconcile_tags_with_words(words, tags, sentence_text, verbose=True):
    """
    Ensure len(tags) == len(words). If mismatch:
      - If tags longer -> truncate (warn)
      - If tags shorter -> pad with KEEP (warn)
    Returns reconciled_tags (length == len(words)) and a flag if reconciliation happened.
    """
    n_words = len(words)
    n_tags = len(tags)
    if n_words == n_tags:
        return tags, False
    if verbose:
        print(f"[RECONCILE] sentence: {sentence_text!r}")
        print(f"[RECONCILE] canonical words ({n_words}): {words}")
        print(f"[RECONCILE] original tags ({n_tags}): {tags}")
    if n_tags > n_words:
        reconciled = tags[:n_words]
        if verbose:
            print(f"[RECONCILE] Warning: tags longer than words -> truncating tags to {n_words} entries.")
    else:
        reconciled = tags + ["KEEP"] * (n_words - n_tags)
        if verbose:
            print(f"[RECONCILE] Warning: tags shorter than words -> padding with KEEP to {n_words} entries.")
    if verbose:
        print(f"[RECONCILE] reconciled tags ({len(reconciled)}): {reconciled}\n")
    return reconciled, True



In [ ]:
# -------------------------
# Example training data (note: tags must align to canonical_word_split)
# -------------------------
train_data = [
    ("She go to school every day .", ["KEEP","REPLACE:goes","KEEP","KEEP","KEEP","KEEP","KEEP"]),
    # The problematic example: ensure tags align to canonical split ["He","have","two","apple","."]
    ("He have two apple .", ["KEEP","REPLACE:has","KEEP","REPLACE:apples","KEEP"])
]

# If your original dataset had 6 tags for the second sentence, reconcile_tags_with_words will show the fix.


In [ ]:
# -------------------------
# Initialize model and optimizer
# -------------------------
model = EditTagger(ENCODER_NAME, num_tags=len(TAG_LIST)).to(DEVICE)
model.train()
optimizer = torch.optim.Adam(model.classifier.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()



In [ ]:
# -------------------------
# Build training tensors with reconciliation
# -------------------------
X = []
Y = []
for sent, tags in train_data:
    words = canonical_word_split(sent)
    # Reconcile tags if needed (prints warning if mismatch)
    tags_reconciled, changed = reconcile_tags_with_words(words, tags, sent, verbose=True)
    # Now get embeddings (guaranteed length == len(words))
    wvecs = get_word_embeddings_canonical(sent, model)   # (num_words, H)
    if wvecs.size(0) != len(tags_reconciled):
        raise RuntimeError("Internal mismatch after reconciliation; aborting.")
    for i, tag in enumerate(tags_reconciled):
        X.append(wvecs[i])
        Y.append(TAG2IDX[tag])
X = torch.stack(X).to(DEVICE)
Y = torch.tensor(Y).to(DEVICE)

# Train head (demo)
for epoch in range(60):
    model.classifier.train()
    optimizer.zero_grad()
    logits = model.classifier(X)
    loss = loss_fn(logits, Y)
    loss.backward()
    optimizer.step()
model.eval()



In [ ]:
# -------------------------
# Quick inference to show pipeline for the problematic sentence
# -------------------------
test_sent = "He have two apple ."
words, enc, word_ids = tokenize_with_alignment_canonical(test_sent)
print("Canonical words:", words)
print("word_ids mapping:", word_ids)

input_ids = enc["input_ids"].to(DEVICE)
attention_mask = enc["attention_mask"].to(DEVICE)
with torch.no_grad():
    logits_subword = model(input_ids=input_ids, attention_mask=attention_mask)[0]

word_logits = aggregate_subword_logits_to_words(logits_subword, word_ids, num_words=len(words))
word_probs = [F.softmax(lg, dim=-1).cpu().numpy() for lg in word_logits]
pred_idxs = [int(np.argmax(p)) for p in word_probs]
pred_tags = [IDX2TAG[idx] for idx in pred_idxs]

print("Predicted tags:", pred_tags)
# Apply tags (simple apply_tags from earlier; implement or reuse)
def apply_tags(words, tags):
    out = []
    for w, tag in zip(words, tags):
        if tag == "KEEP":
            out.append(w)
        elif tag == "DELETE":
            continue
        elif tag.startswith("REPLACE:"):
            out.append(tag.split("REPLACE:")[1])
        elif tag.startswith("INSERT_BEFORE:"):
            out.append(tag.split("INSERT_BEFORE:")[1])
            out.append(w)
        else:
            out.append(w)
    s = " ".join(out)
    s = s.replace(" .", ".").replace(" ,", ",")
    return s

print("Corrected:", apply_tags(words, pred_tags))

